# DFT Validation of MACE Disorder Predictions

**Goal**: Validate that MACE-MP-0 preserves dopant rankings when comparing ordered vs SQS-disordered LiCoO₂ structures against DFT (Quantum ESPRESSO PBE+U).

**Protocol**:
1. Generate ordered + SQS-disordered structures for 5 dopants (Al, Ti, Mg, Ga, Fe)
2. MACE-relax all structures (GPU)
3. Single-point DFT SCF on MACE-relaxed geometries (CPU)
4. Compare voltage rankings: Spearman ρ (MACE vs DFT)

**Runtime**: Use **T4 GPU** — MACE needs GPU, QE uses CPU cores.

**Estimated time**: ~3-5 hours total (30 min MACE + 2-4 hr QE)

## Cell 1: Install dependencies

In [ ]:
# Install Quantum ESPRESSO (CPU) and MACE (GPU)
!apt-get -qq install quantum-espresso
!pip install -q pymatgen mace-torch numpy scipy ase

# Verify GPU is available for MACE
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify QE is installed
!which pw.x && echo 'QE installed OK'

## Cell 2: Download pseudopotentials (run after Cell 3 clone)

In [ ]:
import os

# Must run AFTER Cell 3 (clone + chdir into repo)
os.makedirs("pseudo", exist_ok=True)

PSEUDO_BASE = "https://pseudopotentials.quantum-espresso.org/upf_files"
pseudos = [
    "Li.pbe-s-kjpaw_psl.1.0.0.UPF",
    "Co.pbe-spn-kjpaw_psl.0.3.1.UPF",
    "O.pbe-n-kjpaw_psl.1.0.0.UPF",
    "Al.pbe-n-kjpaw_psl.1.0.0.UPF",
    "Ti.pbe-spn-kjpaw_psl.1.0.0.UPF",
    "Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF",
    "Ga.pbe-dn-kjpaw_psl.1.0.0.UPF",
    "Fe.pbe-spn-kjpaw_psl.0.2.1.UPF",
]
for p in pseudos:
    !wget -q -nc -P pseudo {PSEUDO_BASE}/{p}

!ls -la pseudo/

## Cell 3: Create parent structure + SQS generator (self-contained, no clone needed)

In [ ]:
import os, pathlib

# Write parent LiCoO2 CIF inline (no repo clone needed)
os.makedirs("data/structures", exist_ok=True)

LCO_CIF = """# generated using pymatgen
data_LiCoO2
_symmetry_space_group_name_H-M   'P 1'
_cell_length_a   2.87500000
_cell_length_b   2.87500000
_cell_length_c   14.20000000
_cell_angle_alpha   90.00000000
_cell_angle_beta   90.00000000
_cell_angle_gamma   120.00000000
_symmetry_Int_Tables_number   1
_chemical_formula_structural   LiCoO2
_chemical_formula_sum   'Li1 Co1 O2'
_cell_volume   101.64702544
_cell_formula_units_Z   1
loop_
 _symmetry_equiv_pos_site_id
 _symmetry_equiv_pos_as_xyz
  1  'x, y, z'
loop_
 _atom_site_type_symbol
 _atom_site_label
 _atom_site_symmetry_multiplicity
 _atom_site_fract_x
 _atom_site_fract_y
 _atom_site_fract_z
 _atom_site_occupancy
  Li  Li0  1  0.00000000  0.00000000  0.50000000  1
  Co  Co1  1  0.00000000  0.00000000  0.00000000  1
  O  O2  1  0.00000000  0.00000000  0.26020000  1
  O  O3  1  0.00000000  0.00000000  0.73980000  1
"""

pathlib.Path("data/structures/lco_parent.cif").write_text(LCO_CIF)
print("Parent CIF written.")
!ls -la data/structures/lco_parent.cif

## Cell 4: Configuration

In [ ]:
import json
import pathlib
import re
import sys
import time

import numpy as np

# ── Parameters ──
ECUTWFC = 40.0       # Ry
ECUTRHO = 320.0      # Ry (8x for PAW)
CONV_THR = 1.0e-5
PSEUDO_DIR = "./pseudo"  # relative to repo root (cwd after Cell 3)

SUPERCELL = [3, 3, 2]  # 72 atoms, 18 Co sites
CONCENTRATION = 0.10   # 10% dopant on Co site
N_SQS = 2             # 2 SQS per dopant (Colab time budget)

DOPANTS = ["Al", "Ti", "Mg", "Ga", "Fe"]

HUBBARD_U = {"Co": 3.32, "Fe": 5.3}  # eV, Materials Project standard

PSEUDOS = {
    "Li": "Li.pbe-s-kjpaw_psl.1.0.0.UPF",
    "Co": "Co.pbe-spn-kjpaw_psl.0.3.1.UPF",
    "O":  "O.pbe-n-kjpaw_psl.1.0.0.UPF",
    "Al": "Al.pbe-n-kjpaw_psl.1.0.0.UPF",
    "Ti": "Ti.pbe-spn-kjpaw_psl.1.0.0.UPF",
    "Mg": "Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF",
    "Ga": "Ga.pbe-dn-kjpaw_psl.1.0.0.UPF",
    "Fe": "Fe.pbe-spn-kjpaw_psl.0.2.1.UPF",
}

MASSES = {
    "Li": 6.941, "Co": 58.933, "O": 15.999,
    "Al": 26.982, "Ti": 47.867, "Mg": 24.305,
    "Ga": 69.723, "Fe": 55.845,
}

DATA_DIR = pathlib.Path("data/structures")
OUT_DIR = pathlib.Path("qe_sqs_inputs")
STRUCT_DIR = pathlib.Path("qe_sqs_structures")
OUT_DIR.mkdir(exist_ok=True)
STRUCT_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"Dopants: {DOPANTS}")
print(f"Supercell: {SUPERCELL} = 72 atoms")
print(f"SQS per dopant: {N_SQS}")

## Cell 5: Helper functions (structure generation, MACE relax, QE input)

In [ ]:
import random
import numpy as np
from pymatgen.core import Structure


# ── SQS generator (self-contained, from stages/stage5/sqs_generator.py) ──

def generate_sqs(parent_structure, dopant_element, target_species,
                 concentration, supercell_matrix, n_realisations=5, n_trials=100):
    """Generate SQS structures via random sampling + pair-correlation scoring.
    n_trials=100 is sufficient for 2 dopants on 18 sites (only 153 unique combos)."""
    supercell = parent_structure.copy()
    supercell.make_supercell(supercell_matrix)

    target_indices = [i for i, site in enumerate(supercell)
                      if site.species_string == target_species]
    n_target = len(target_indices)
    n_dopant = round(concentration * n_target)
    if n_dopant < 1:
        raise ValueError(f"Concentration {concentration} on {n_target} sites = {concentration*n_target:.2f} atoms")

    # Precompute NN distance matrix for target sites (avoid recomputing per trial)
    lattice = supercell.lattice
    nn_pairs = []  # list of (i_local, j_local) pairs within 5 Å
    for i in range(n_target):
        for j in range(i+1, n_target):
            dist = lattice.get_distance_and_image(
                supercell[target_indices[i]].frac_coords,
                supercell[target_indices[j]].frac_coords)[0]
            if dist < 5.0:
                nn_pairs.append((i, j))
    print(f"    SQS: {n_target} target sites, {n_dopant} dopants, {len(nn_pairs)} NN pairs")

    structures = []
    seen = set()
    for r in range(n_realisations):
        best_score = float("inf")
        best_struct = None
        best_chosen = None
        for t in range(n_trials):
            trial = supercell.copy()
            chosen = frozenset(random.sample(target_indices, n_dopant))
            for idx in chosen:
                trial.replace(idx, dopant_element)

            # Fast scoring using precomputed pairs
            chosen_local = {i for i, tidx in enumerate(target_indices) if tidx in chosen}
            score = _fast_pair_corr_dev(chosen_local, nn_pairs, n_dopant / n_target)

            if score < best_score or (score == best_score and chosen not in seen):
                best_score = score
                best_struct = trial
                best_chosen = chosen

        seen.add(best_chosen)
        structures.append(best_struct)
        print(f"    SQS realisation {r}: score={best_score:.6f}")
    return structures


def _fast_pair_corr_dev(chosen_local, nn_pairs, c):
    """Fast pair correlation deviation using precomputed NN pairs."""
    dd, dh, total = 0, 0, len(nn_pairs)
    if total == 0:
        return 0.0
    for i, j in nn_pairs:
        i_dop = i in chosen_local
        j_dop = j in chosen_local
        if i_dop and j_dop:
            dd += 1
        elif i_dop != j_dop:
            dh += 1
    return (dd/total - c*c)**2 + (dh/total - 2*c*(1-c))**2


# ── Structure builders ──

def build_parent_supercell():
    struct = Structure.from_file(str(DATA_DIR / "lco_parent.cif"))
    struct.make_supercell(SUPERCELL)
    print(f"Parent supercell: {len(struct)} atoms")
    return struct


def build_ordered_structure(parent, dopant):
    struct = parent.copy()
    co_indices = [i for i, sp in enumerate(struct.species) if str(sp) == "Co"]
    n_dopant = round(CONCENTRATION * len(co_indices))
    for idx in co_indices[:n_dopant]:
        struct.replace(idx, dopant)
    return struct


def remove_li(struct):
    s = struct.copy()
    li_idx = sorted([i for i, sp in enumerate(s.species) if str(sp) == "Li"], reverse=True)
    for idx in li_idx:
        s.remove_sites([idx])
    return s


# ── MACE relaxation (cached calculator) ──

_mace_calc = None

def mace_relax(struct, fmax=0.15, steps=500):
    global _mace_calc
    from mace.calculators import mace_mp
    from ase.optimize import BFGS
    from pymatgen.io.ase import AseAtomsAdaptor

    if _mace_calc is None:
        _mace_calc = mace_mp(model="medium", default_dtype="float64", device="cuda")

    atoms = AseAtomsAdaptor.get_atoms(struct)
    atoms.calc = _mace_calc
    opt = BFGS(atoms, logfile=None)
    opt.run(fmax=fmax, steps=steps)
    relaxed = AseAtomsAdaptor.get_structure(atoms)
    return relaxed, atoms.get_potential_energy()


# ── QE input generation ──

def struct_to_qe_input(struct, prefix):
    lattice = struct.lattice
    species_list = [str(sp) for sp in struct.species]
    unique_species = sorted(set(species_list))
    nat, ntyp = len(struct), len(unique_species)

    u_species = {sp: HUBBARD_U[sp] for sp in unique_species
                 if sp in HUBBARD_U and HUBBARD_U[sp] > 0}

    lines = []
    lines.append("&CONTROL")
    lines.append(f"  calculation = 'scf'")
    lines.append(f"  prefix = '{prefix}'")
    lines.append(f"  outdir = './tmp_{prefix}'")
    lines.append(f"  pseudo_dir = '{PSEUDO_DIR}'")
    lines.append(f"  tprnfor = .false.")
    lines.append(f"  tstress = .false.")
    lines.append(f"  verbosity = 'low'")
    lines.append(f"  disk_io = 'low'")
    lines.append("/")
    lines.append("")
    lines.append("&SYSTEM")
    lines.append(f"  ibrav = 0")
    lines.append(f"  nat = {nat}")
    lines.append(f"  ntyp = {ntyp}")
    lines.append(f"  ecutwfc = {ECUTWFC}")
    lines.append(f"  ecutrho = {ECUTRHO}")
    lines.append(f"  occupations = 'smearing'")
    lines.append(f"  smearing = 'mv'")
    lines.append(f"  degauss = 0.02")
    lines.append(f"  nspin = 2")
    for i, sp in enumerate(unique_species):
        if sp in ("Co", "Fe", "Ti"):
            lines.append(f"  starting_magnetization({i + 1}) = 0.5")
    if u_species:
        lines.append(f"  lda_plus_u = .true.")
        for i, sp in enumerate(unique_species):
            if sp in u_species:
                lines.append(f"  Hubbard_U({i + 1}) = {u_species[sp]}")
    lines.append("/")
    lines.append("")
    lines.append("&ELECTRONS")
    lines.append(f"  conv_thr = {CONV_THR}")
    lines.append(f"  mixing_mode = 'local-TF'")
    lines.append(f"  mixing_beta = 0.1")
    lines.append(f"  electron_maxstep = 500")
    lines.append(f"  diagonalization = 'david'")
    lines.append("/")
    lines.append("")
    lines.append("CELL_PARAMETERS angstrom")
    for vec in lattice.matrix:
        lines.append(f"  {vec[0]:16.10f} {vec[1]:16.10f} {vec[2]:16.10f}")
    lines.append("")
    lines.append("ATOMIC_SPECIES")
    for sp in unique_species:
        lines.append(f"  {sp:4s} {MASSES.get(sp, 1.0):10.4f} {PSEUDOS[sp]}")
    lines.append("")
    lines.append("ATOMIC_POSITIONS crystal")
    for sp, site in zip(species_list, struct.sites):
        fc = site.frac_coords
        lines.append(f"  {sp:4s} {fc[0]:16.10f} {fc[1]:16.10f} {fc[2]:16.10f}")
    lines.append("")
    lines.append("K_POINTS gamma")
    lines.append("")
    return "\n".join(lines)


def generate_li_metal_input():
    return f"""&CONTROL
  calculation = 'scf'
  prefix = 'li_metal'
  outdir = './tmp_li_metal'
  pseudo_dir = '{PSEUDO_DIR}'
  tprnfor = .false.
  tstress = .false.
  verbosity = 'low'
  disk_io = 'low'
/

&SYSTEM
  ibrav = 0
  nat = 2
  ntyp = 1
  ecutwfc = {ECUTWFC}
  ecutrho = {ECUTRHO}
  occupations = 'smearing'
  smearing = 'mv'
  degauss = 0.02
/

&ELECTRONS
  conv_thr = {CONV_THR}
  mixing_beta = 0.7
  electron_maxstep = 100
  diagonalization = 'david'
/

CELL_PARAMETERS angstrom
  3.4900000000   0.0000000000   0.0000000000
  0.0000000000   3.4900000000   0.0000000000
  0.0000000000   0.0000000000   3.4900000000

ATOMIC_SPECIES
  Li   {MASSES['Li']:10.4f} {PSEUDOS['Li']}

ATOMIC_POSITIONS crystal
  Li    0.0000000000   0.0000000000   0.0000000000
  Li    0.5000000000   0.5000000000   0.5000000000

K_POINTS automatic
  8 8 8 0 0 0
"""


print("All helper functions loaded.")

## Cell 6: Generate structures + MACE relax (GPU, ~30 min)

This cell builds all structures, relaxes them with MACE on GPU, and writes QE input files.

In [ ]:
t_start = time.time()

parent = build_parent_supercell()
manifest = {}  # name -> {dopant, type, mace_energy_eV, n_li, n_atoms}
all_calc_names = []

# ── Undoped lithiated ──
print("\n--- Undoped lithiated (LiCoO2) ---")
lith_r, lith_e = mace_relax(parent.copy())
n_li = sum(1 for sp in lith_r.species if str(sp) == "Li")
name = "undoped_lith"
(OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(lith_r, name))
lith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
manifest[name] = {"dopant": "none", "type": "undoped_lith",
                   "mace_energy_eV": lith_e, "n_li": n_li, "n_atoms": len(lith_r)}
all_calc_names.append(name)
print(f"  E_MACE = {lith_e:.4f} eV, n_Li = {n_li}")

# ── Undoped delithiated ──
print("--- Undoped delithiated (CoO2) ---")
delith_r, delith_e = mace_relax(remove_li(parent))
name = "undoped_delith"
(OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(delith_r, name))
delith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
manifest[name] = {"dopant": "none", "type": "undoped_delith",
                   "mace_energy_eV": delith_e, "n_li": 0, "n_atoms": len(delith_r)}
all_calc_names.append(name)
print(f"  E_MACE = {delith_e:.4f} eV")

# ── Li metal ──
print("--- Li metal (bcc) ---")
name = "li_metal"
(OUT_DIR / f"{name}.in").write_text(generate_li_metal_input())
manifest[name] = {"dopant": "none", "type": "li_metal",
                   "mace_energy_eV": None, "n_li": 2, "n_atoms": 2}
all_calc_names.append(name)

# ── Per-dopant ──
for dopant in DOPANTS:
    print(f"\n=== {dopant} ===")
    t0 = time.time()

    # Ordered lithiated
    ord_lith = build_ordered_structure(parent, dopant)
    ord_lith_r, ord_lith_e = mace_relax(ord_lith)
    n_li = sum(1 for sp in ord_lith_r.species if str(sp) == "Li")
    name = f"{dopant}_ord_lith"
    (OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(ord_lith_r, name))
    ord_lith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
    manifest[name] = {"dopant": dopant, "type": "ordered_lith",
                       "mace_energy_eV": ord_lith_e, "n_li": n_li, "n_atoms": len(ord_lith_r)}
    all_calc_names.append(name)

    # Ordered delithiated
    ord_delith_r, ord_delith_e = mace_relax(remove_li(build_ordered_structure(parent, dopant)))
    name = f"{dopant}_ord_delith"
    (OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(ord_delith_r, name))
    ord_delith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
    manifest[name] = {"dopant": dopant, "type": "ordered_delith",
                       "mace_energy_eV": ord_delith_e, "n_li": 0, "n_atoms": len(ord_delith_r)}
    all_calc_names.append(name)

    # SQS disordered
    sqs_structs = generate_sqs(
        parent_structure=parent.copy(),
        dopant_element=dopant,
        target_species="Co",
        concentration=CONCENTRATION,
        supercell_matrix=SUPERCELL,
        n_realisations=N_SQS,
    )
    for i, sqs in enumerate(sqs_structs):
        # Lithiated
        sqs_lith_r, sqs_lith_e = mace_relax(sqs.copy())
        n_li_sqs = sum(1 for sp in sqs_lith_r.species if str(sp) == "Li")
        name = f"{dopant}_sqs{i}_lith"
        (OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(sqs_lith_r, name))
        sqs_lith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
        manifest[name] = {"dopant": dopant, "type": f"sqs{i}_lith",
                           "mace_energy_eV": sqs_lith_e, "n_li": n_li_sqs, "n_atoms": len(sqs_lith_r)}
        all_calc_names.append(name)

        # Delithiated
        sqs_delith_r, sqs_delith_e = mace_relax(remove_li(sqs))
        name = f"{dopant}_sqs{i}_delith"
        (OUT_DIR / f"{name}.in").write_text(struct_to_qe_input(sqs_delith_r, name))
        sqs_delith_r.to(filename=str(STRUCT_DIR / f"{name}.cif"))
        manifest[name] = {"dopant": dopant, "type": f"sqs{i}_delith",
                           "mace_energy_eV": sqs_delith_e, "n_li": 0, "n_atoms": len(sqs_delith_r)}
        all_calc_names.append(name)

    print(f"  {dopant} done in {time.time()-t0:.0f}s")

# Save manifest
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Save calc order
with open(OUT_DIR / "calc_names.json", "w") as f:
    json.dump(all_calc_names, f)

t_mace = time.time() - t_start
print(f"\n{'='*60}")
print(f"  MACE relaxation complete: {len(all_calc_names)} structures in {t_mace:.0f}s")
print(f"  QE inputs written to: {OUT_DIR}/")
print(f"  Structures saved to: {STRUCT_DIR}/")
print(f"{'='*60}")

## Cell 7: Run QE calculations (CPU, ~2-4 hours)

Runs all single-point SCF calculations sequentially. Each 72-atom calc takes ~5-10 min on Colab CPU.

In [ ]:
import subprocess

with open(OUT_DIR / "calc_names.json") as f:
    all_calc_names = json.load(f)

t_start = time.time()
results_summary = []

for idx, name in enumerate(all_calc_names):
    print(f"[{idx+1}/{len(all_calc_names)}] {name} ...", end=" ", flush=True)
    t0 = time.time()

    # Create tmp dir
    os.makedirs(f"tmp_{name}", exist_ok=True)

    # Run pw.x
    in_file = OUT_DIR / f"{name}.in"
    out_file = OUT_DIR / f"{name}.out"
    result = subprocess.run(
        ["pw.x"],
        stdin=open(in_file),
        stdout=open(out_file, "w"),
        stderr=subprocess.STDOUT,
        timeout=3600,  # 1 hour max per calc
    )

    elapsed = time.time() - t0

    # Check convergence
    out_text = out_file.read_text()
    converged = bool(re.search(r"!\s+total energy", out_text))
    not_converged = "convergence NOT achieved" in out_text

    status = "OK" if converged else ("NOT CONVERGED" if not_converged else "ERROR")
    print(f"{status} ({elapsed:.0f}s)")
    results_summary.append({"name": name, "status": status, "time_s": elapsed})

t_total = time.time() - t_start
n_ok = sum(1 for r in results_summary if r["status"] == "OK")
print(f"\n{'='*60}")
print(f"  QE complete: {n_ok}/{len(all_calc_names)} converged in {t_total:.0f}s ({t_total/60:.0f} min)")
print(f"{'='*60}")

# Show any failures
for r in results_summary:
    if r["status"] != "OK":
        print(f"  FAILED: {r['name']} ({r['status']})")

## Cell 8: Parse results and compute Spearman rho

In [ ]:
from scipy.stats import spearmanr

RY_TO_EV = 13.605698

def parse_qe_energy(name):
    """Extract total energy (Ry) from QE output."""
    out_file = OUT_DIR / f"{name}.out"
    if not out_file.exists():
        return None
    text = out_file.read_text()
    if "convergence NOT achieved" in text:
        print(f"  WARNING: {name} did NOT converge")
    matches = re.findall(r"!\s+total energy\s+=\s+([-\d.]+)\s+Ry", text)
    return float(matches[-1]) if matches else None


def compute_voltage(e_lith_ry, e_delith_ry, n_li, e_li_per_atom_ry):
    """V = -(E_lith - E_delith - n_Li * E_Li_metal) / n_Li  [eV]"""
    return -(e_lith_ry - e_delith_ry - n_li * e_li_per_atom_ry) * RY_TO_EV / n_li


# Load manifest
with open(OUT_DIR / "manifest.json") as f:
    manifest = json.load(f)

# Li metal reference
e_li_metal = parse_qe_energy("li_metal")
if e_li_metal is None:
    raise RuntimeError("Li metal energy not found!")
e_li_per_atom = e_li_metal / 2.0
print(f"Li metal: {e_li_metal:.8f} Ry ({e_li_per_atom:.8f} Ry/atom)")

# Undoped reference
e_undoped_lith = parse_qe_energy("undoped_lith")
e_undoped_delith = parse_qe_energy("undoped_delith")
if e_undoped_lith and e_undoped_delith:
    n_li_ref = manifest["undoped_lith"]["n_li"]
    v_undoped = compute_voltage(e_undoped_lith, e_undoped_delith, n_li_ref, e_li_per_atom)
    print(f"Undoped LiCoO2 voltage: {v_undoped:.4f} V (exp: ~3.9 V)")

# Per-dopant
results = {}
for dopant in DOPANTS:
    print(f"\n--- {dopant} ---")
    res = {"dopant": dopant}

    # Ordered (DFT)
    e_ol = parse_qe_energy(f"{dopant}_ord_lith")
    e_od = parse_qe_energy(f"{dopant}_ord_delith")
    if e_ol and e_od:
        n_li = manifest[f"{dopant}_ord_lith"]["n_li"]
        res["v_ord_dft"] = compute_voltage(e_ol, e_od, n_li, e_li_per_atom)
        print(f"  Ordered DFT: {res['v_ord_dft']:.4f} V")
    else:
        res["v_ord_dft"] = None

    # Ordered (MACE)
    e_ol_m = manifest[f"{dopant}_ord_lith"]["mace_energy_eV"]
    e_od_m = manifest[f"{dopant}_ord_delith"]["mace_energy_eV"]
    n_li = manifest[f"{dopant}_ord_lith"]["n_li"]
    res["v_ord_mace"] = -(e_ol_m - e_od_m) / n_li

    # SQS (DFT + MACE)
    sqs_v_dft = []
    sqs_v_mace = []
    for i in range(N_SQS):
        e_sl = parse_qe_energy(f"{dopant}_sqs{i}_lith")
        e_sd = parse_qe_energy(f"{dopant}_sqs{i}_delith")
        if e_sl and e_sd:
            n_li_s = manifest[f"{dopant}_sqs{i}_lith"]["n_li"]
            v = compute_voltage(e_sl, e_sd, n_li_s, e_li_per_atom)
            sqs_v_dft.append(v)
            print(f"  SQS{i} DFT: {v:.4f} V")

        e_sl_m = manifest[f"{dopant}_sqs{i}_lith"]["mace_energy_eV"]
        e_sd_m = manifest[f"{dopant}_sqs{i}_delith"]["mace_energy_eV"]
        n_li_s = manifest[f"{dopant}_sqs{i}_lith"]["n_li"]
        sqs_v_mace.append(-(e_sl_m - e_sd_m) / n_li_s)

    res["v_dis_dft"] = float(np.mean(sqs_v_dft)) if sqs_v_dft else None
    res["v_dis_dft_std"] = float(np.std(sqs_v_dft)) if sqs_v_dft else None
    res["v_dis_mace"] = float(np.mean(sqs_v_mace))
    res["sqs_v_dft"] = sqs_v_dft
    res["sqs_v_mace"] = sqs_v_mace
    results[dopant] = res

# ── Spearman correlations ──
print(f"\n{'='*60}")
print("  RANKING COMPARISON: MACE vs DFT")
print(f"{'='*60}")

spearman_results = {}

# 1. Ordered
m, d, lab = [], [], []
for dop in DOPANTS:
    r = results[dop]
    if r["v_ord_dft"] is not None:
        m.append(r["v_ord_mace"]); d.append(r["v_ord_dft"]); lab.append(dop)
if len(m) >= 3:
    rho, p = spearmanr(m, d)
    spearman_results["ordered"] = {"rho": rho, "p": p, "n": len(m)}
    print(f"\n  Ordered (MACE vs DFT), n={len(m)}: rho={rho:.3f}, p={p:.3f}")
    for i, lb in enumerate(lab):
        print(f"    {lb:4s}: MACE={m[i]:.4f}  DFT={d[i]:.4f}")

# 2. Disordered
m, d, lab = [], [], []
for dop in DOPANTS:
    r = results[dop]
    if r["v_dis_dft"] is not None:
        m.append(r["v_dis_mace"]); d.append(r["v_dis_dft"]); lab.append(dop)
if len(m) >= 3:
    rho, p = spearmanr(m, d)
    spearman_results["disordered"] = {"rho": rho, "p": p, "n": len(m)}
    print(f"\n  Disordered (MACE vs DFT), n={len(m)}: rho={rho:.3f}, p={p:.3f}")
    for i, lb in enumerate(lab):
        print(f"    {lb:4s}: MACE={m[i]:.4f}  DFT={d[i]:.4f}")

# 3. Disorder-induced shift (KEY METRIC)
m, d, lab = [], [], []
for dop in DOPANTS:
    r = results[dop]
    if r["v_ord_dft"] is not None and r["v_dis_dft"] is not None:
        m.append(r["v_dis_mace"] - r["v_ord_mace"])
        d.append(r["v_dis_dft"] - r["v_ord_dft"])
        lab.append(dop)
if len(m) >= 3:
    rho, p = spearmanr(m, d)
    spearman_results["disorder_shift"] = {"rho": rho, "p": p, "n": len(m)}
    print(f"\n  Disorder-induced shift (MACE vs DFT), n={len(m)}: rho={rho:.3f}, p={p:.3f}")
    print(f"  >>> THIS IS THE KEY VALIDATION METRIC <<<")
    for i, lb in enumerate(lab):
        print(f"    {lb:4s}: MACE shift={m[i]:+.4f}  DFT shift={d[i]:+.4f}")

# Save
output = {
    "dopants": DOPANTS,
    "supercell": SUPERCELL,
    "n_sqs": N_SQS,
    "concentration": CONCENTRATION,
    "ecutwfc_Ry": ECUTWFC,
    "li_metal_Ry_per_atom": e_li_per_atom,
    "undoped_voltage_V": v_undoped if (e_undoped_lith and e_undoped_delith) else None,
    "per_dopant": results,
    "spearman": spearman_results,
}
with open("paper/dft_sqs_validation_results.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"\nResults saved to paper/dft_sqs_validation_results.json")

## Cell 9: Download results

In [ ]:
# Download results JSON to local machine
from google.colab import files

files.download("paper/dft_sqs_validation_results.json")
files.download(str(OUT_DIR / "manifest.json"))

# Also zip the QE outputs for archival
!zip -q dft_sqs_outputs.zip qe_sqs_inputs/*.out qe_sqs_structures/*.cif
files.download("dft_sqs_outputs.zip")

print("Downloads complete.")